LOADING THE DATA

In [1]:
#Importing 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split

In [2]:
#loading fake_dataset
with open('../data/data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

fake_df1 = pd.DataFrame(data)
fake_df2=pd.read_csv('../data/synthetic_fake_reviews.csv',)




In [3]:
#loading genuine_dataset
genuine_df1=pd.read_csv('../data/G_review1.csv')
genuine_df2=pd.read_csv('../data/G_review2.csv')


In [4]:
#loading real data 
df_real=pd.read_csv('../data/theultimatefinal.csv')
print(f"Real scraped reviews : {len(df_real)}")
print(df_real.columns.tolist())




Real scraped reviews : 960
['review_text', 'review_date', 'rating', 'language_style', 'has_slang', 'slang_count']


In [5]:
#importing reusable functions
import sys
import os

# Add src folder to Python path
sys.path.append(os.path.abspath("../src"))


from pre import get_length_type, add_feature_columns,generate_date,detect_slang

In [6]:
fake_df2.head()


,review_id,review_text,label,fake_type,language_style,domain,length_type,word_count,synthetic
0,SYN_00001,"Dherai kharaab service, I hate this salon, sta...",fake,negative_attack,code_mixed,services,very_short,12,True
1,SYN_00002,"Bank ko customer care bhayaanak thiyo, I didn'...",fake,negative_attack,code_mixed,services,very_short,14,True
2,SYN_00003,"Repair shop ko price dherai high cha, I won't ...",fake,negative_attack,code_mixed,services,very_short,13,True
3,SYN_00004,"Hospital ko doctor haraamkhor thiye, I won't g...",fake,negative_attack,code_mixed,services,very_short,12,True
4,SYN_00005,"Salon ko hair stylist bhayaanak skilled thiyo,...",fake,negative_attack,code_mixed,services,very_short,14,True


In [7]:

genuine_df = pd.concat([genuine_df1, genuine_df2], ignore_index=True)


In [8]:

fake_df1['word_count'] = fake_df1['review_text'].apply(lambda x: len(str(x).split()))
genuine_df['word_count'] = genuine_df['review_text'].apply(lambda x: len(str(x).split()))


In [9]:
fake_df1["length_type"] = fake_df1["word_count"].apply(get_length_type)
genuine_df["length_type"] = genuine_df["word_count"].apply(get_length_type)

In [10]:
# Drop multiple columns
fake_df2.drop(columns=['review_id', 'label', 'synthetic','length_type'], inplace=True)
fake_df2.head()

,review_text,fake_type,language_style,domain,word_count
0,"Dherai kharaab service, I hate this salon, sta...",negative_attack,code_mixed,services,12
1,"Bank ko customer care bhayaanak thiyo, I didn'...",negative_attack,code_mixed,services,14
2,"Repair shop ko price dherai high cha, I won't ...",negative_attack,code_mixed,services,13
3,"Hospital ko doctor haraamkhor thiye, I won't g...",negative_attack,code_mixed,services,12
4,"Salon ko hair stylist bhayaanak skilled thiyo,...",negative_attack,code_mixed,services,14


In [11]:
fake_df1.head()

,review_text,rating,date,language_style,has_slang,slang_count,domain,fake_type,word_count,length_type
0,"yo product lastai ramro cha, life ma yesto qua...",5,2023-07-12,romanized_nepali,1.0,2.0,product,NaN,14,short
1,This hotel is absolutely the best in the entir...,5,2024-03-22,english,0.0,0.0,hotel_restaurant,NaN,19,short
2,food ekdum dami cha ani service pani world cla...,5,2023-11-08,code_mixed,1.0,1.0,food,NaN,15,short
3,"worst motorcycle ever, ekdum kharab, paisa was...",1,2022-09-15,code_mixed,0.0,0.0,transportation,NaN,11,short
4,"yo service sabai vanda ramro, ekdum profession...",5,2024-01-30,romanized_nepali,1.0,1.0,services,NaN,12,short


In [12]:
fake_df2["length_type"] = fake_df2["word_count"].apply(get_length_type)
fake_df2 = add_feature_columns(fake_df2)
fake_df2.head()


,review_text,fake_type,language_style,domain,word_count,length_type,rating,date,has_slang,slang_count
0,"Dherai kharaab service, I hate this salon, sta...",negative_attack,code_mixed,services,12,short,1,2022-10-16,0,0
1,"Bank ko customer care bhayaanak thiyo, I didn'...",negative_attack,code_mixed,services,14,short,1,2023-11-08,0,0
2,"Repair shop ko price dherai high cha, I won't ...",negative_attack,code_mixed,services,13,short,1,2023-10-26,0,0
3,"Hospital ko doctor haraamkhor thiye, I won't g...",negative_attack,code_mixed,services,12,short,1,2024-04-29,0,0
4,"Salon ko hair stylist bhayaanak skilled thiyo,...",negative_attack,code_mixed,services,14,short,1,2024-08-27,0,0


In [13]:
fake_df = pd.concat([fake_df1, fake_df2], ignore_index=True)
fake_df.head()

,review_text,rating,date,language_style,has_slang,slang_count,domain,fake_type,word_count,length_type
0,"yo product lastai ramro cha, life ma yesto qua...",5,2023-07-12,romanized_nepali,1.0,2.0,product,NaN,14,short
1,This hotel is absolutely the best in the entir...,5,2024-03-22,english,0.0,0.0,hotel_restaurant,NaN,19,short
2,food ekdum dami cha ani service pani world cla...,5,2023-11-08,code_mixed,1.0,1.0,food,NaN,15,short
3,"worst motorcycle ever, ekdum kharab, paisa was...",1,2022-09-15,code_mixed,0.0,0.0,transportation,NaN,11,short
4,"yo service sabai vanda ramro, ekdum profession...",5,2024-01-30,romanized_nepali,1.0,1.0,services,NaN,12,short


In [14]:
# dropping fake_type col before merging with genuine_df
fake_df.drop(columns=["fake_type"], inplace=True)

In [15]:
#selecting some part of syn fake data to act as real fake data 
# Extract ONLY fake reviews
df_syn_fake = fake_df.copy().reset_index(drop=True)
print(f"Total synthetic fake : {len(df_syn_fake)}")
# ── Sample to match real genuine count ───────────────────────
n_real = len(df_real)
# Best quality synthetic fakes = short + medium length
# (more realistic than very_short)
df_syn_fake_quality = df_syn_fake[
    df_syn_fake["length_type"].isin(["short", "medium", "long"])
]

if len(df_syn_fake_quality) >= n_real:
    df_syn_fake_sampled = df_syn_fake_quality.sample(
        n=n_real, random_state=42
    )
else:
    # fallback — use all fake including very_short
    df_syn_fake_sampled = df_syn_fake.sample(
        n=n_real, random_state=42
    )

print(f"Sampled synthetic fake : {len(df_syn_fake_sampled)}")
print(df_syn_fake_sampled["length_type"].value_counts())



Total synthetic fake : 2374
Sampled synthetic fake : 960
length_type
short     674
medium    213
long       73
Name: count, dtype: int64


In [16]:
df_syn_fake_sampled['label']=1

In [17]:
#labeling genuine reviews as 0 and fake reviews as 1
genuine_df['label'] = 0
fake_df['label'] = 1

In [18]:
#merging fake_df and genuine_df
df = pd.concat([genuine_df, fake_df], ignore_index=True)
df.head()

,review_text,rating,date,language_style,has_slang,slang_count,domain,word_count,length_type,label
0,Kwati soup traditional restaurant — 9 types be...,4,10/15/2023,english,0.0,0.0,food,33,medium,0
1,Fuel gauge inaccurate wrong reading,2,12/19/2023,english,0.0,0.0,transportation,5,very_short,0
2,Gym ma personal training session 5 wota try ga...,4,8/2/2024,code_mixed,0.0,0.0,services,76,long,0
3,Tailor worth it ... Service was a bit slow but...,5,11/21/2024,english,0.0,0.0,services,11,short,0
4,Sahayatri ma peak time 😐😊😍 surcharge dherai bh...,3,8/21/2022,romanized_nepali,0.0,0.0,transportation,17,short,0


In [19]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle the dataset
df.head()

,review_text,rating,date,language_style,has_slang,slang_count,domain,word_count,length_type,label
0,MoMo very disappointed with it k hai Battery b...,2,1/7/2022,code_mixed,1.0,1.0,food,17,short,0
1,Dherai ghatiya service rahecha tyo thau ko aaja.,1,2026-02-18,romanized_nepali,1.0,1.0,services,8,very_short,1
2,Bed ramro experience vayo hai ni Malai ta ekda...,5,10/12/2023,romanized_nepali,1.0,1.0,hotel_restaurant,49,medium,0
3,"hotl breakfast timng so short, most itms finis...",1,2024-06-19,english,0.0,0.0,hotel_restaurant,10,very_short,1
4,Katti roll momo vayo thikai cha Kasto ramro th...,3,12/16/2022,romanized_nepali,1.0,1.0,food,24,short,0


In [20]:
#checking for duplicates 
df.duplicated().sum()

np.int64(64)

In [21]:
#checking null values
df.isnull().sum()

review_text        0
rating             0
date              48
language_style     0
has_slang         69
slang_count       69
domain            20
word_count         0
length_type        0
label              0
dtype: int64

In [22]:
#dropping duplicates
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [23]:
#to check count of each unique value in has_slang column
df['has_slang'].value_counts()

has_slang
0.0    2607
1.0    1818
Name: count, dtype: int64

In [24]:
#fixing null val in date

df["date"] = df["date"].fillna(df.apply(lambda _: generate_date(), axis=1))
df.date.isnull().sum()


np.int64(0)

In [25]:
df_syn_fake_sampled["date"]=df_syn_fake_sampled["date"].fillna(df_syn_fake_sampled.apply(lambda _: generate_date(),axis=1))
df_syn_fake_sampled.date.isnull().sum()

np.int64(0)

In [26]:
#only fixing null val in has_slang column
null_slang_mask = df["has_slang"].isnull()
slang_results   = df.loc[null_slang_mask, "review_text"].apply(detect_slang)
df.loc[null_slang_mask, "has_slang"]   = slang_results.apply(lambda x: x[0]).astype(int)
df.loc[null_slang_mask, "slang_count"] = slang_results.apply(lambda x: x[1]).astype(int)
df.has_slang.isnull().sum()

np.int64(0)

In [27]:
null_slang_mask_ = df_syn_fake_sampled["has_slang"].isnull()
slang_results   = df_syn_fake_sampled.loc[null_slang_mask_, "review_text"].apply(detect_slang)
df_syn_fake_sampled.loc[null_slang_mask_, "has_slang"]   = slang_results.apply(lambda x: x[0]).astype(int)
df_syn_fake_sampled.loc[null_slang_mask_, "slang_count"] = slang_results.apply(lambda x: x[1]).astype(int)
df_syn_fake_sampled.has_slang.isnull().sum()

np.int64(0)

In [28]:
df = df.dropna(subset=["domain"])
df.isnull().sum()

review_text       0
rating            0
date              0
language_style    0
has_slang         0
slang_count       0
domain            0
word_count        0
length_type       0
label             0
dtype: int64

In [29]:
df_syn_fake_sampled = df_syn_fake_sampled.dropna(subset=["domain"])
df_syn_fake_sampled.isnull().sum()

review_text       0
rating            0
date              0
language_style    0
has_slang         0
slang_count       0
domain            0
word_count        0
length_type       0
label             0
dtype: int64

In [30]:
df_syn_fake_sampled.drop(columns=["domain"], inplace=True)

In [31]:
df_real.rename(columns={'review_date': 'date'}, inplace=True)


In [32]:
df.domain.value_counts()

domain
services            956
food                920
product             855
hotel_restaurant    802
transportation      763
products            102
hotel                76
Name: count, dtype: int64

In [33]:
#solving category inconsistency issue in domain column
domain_mapping = {
    "products": "product",
    "hotel": "hotel_restaurant"
}

df["domain"] = df["domain"].replace(domain_mapping)
df.domain.value_counts()

domain
product             957
services            956
food                920
hotel_restaurant    878
transportation      763
Name: count, dtype: int64

In [34]:
df_real["label"] = 0    # 0 = genuine
# ── word_count ────────────────────────────────────────────────
df_real["word_count"] = df_real["review_text"].str.split().str.len()

# ── length_type ───────────────────────────────────────────────
df_real["length_type"] = df_real["word_count"].apply(get_length_type)


In [35]:
#combined the genuine and fake data (real data )
df_finetune = pd.concat([
    df_real,
    df_syn_fake_sampled
], ignore_index=True)


FEATURE ENGINEEERING 

In [36]:
# 2. Metadata features (for all models)
meta_features = [
    "word_count",      # review length
    "rating",          # star rating
    "slang_count",     # slang usage
    "has_slang",       # binary slang flag
]

# 3. Engineered features
df["char_count"]        = df["review_text"].str.len()
df["avg_word_len"]      = df["char_count"] / df["word_count"]
df["exclamation_count"] = df["review_text"].str.count("!")
df["question_count"]    = df["review_text"].str.count(r"\?")
df["uppercase_ratio"]   = df["review_text"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)
df["unique_word_ratio"] = df["review_text"].apply(
    lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0
)

In [37]:
df_finetune["char_count"]        = df_finetune["review_text"].str.len()
df_finetune["avg_word_len"]      = df_finetune["char_count"] / df_finetune["word_count"]
df_finetune["exclamation_count"] = df_finetune["review_text"].str.count("!")
df_finetune["question_count"]    = df_finetune["review_text"].str.count(r"\?")
df_finetune["uppercase_ratio"]   = df_finetune["review_text"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)
df_finetune["unique_word_ratio"] = df_finetune["review_text"].apply(
    lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0
)

In [38]:
# ── Shuffle ───────────────────────────────────────────────────
df_finetune = df_finetune.sample(
    frac=1, random_state=42
).reset_index(drop=True)

In [39]:
# ── Drop duplicates ───────────────────────────────────────────
before = len(df_finetune)
df_finetune = df_finetune.drop_duplicates(
    subset=["review_text"], keep="first"
).reset_index(drop=True)
print(f"Duplicates dropped : {before - len(df_finetune)}")

Duplicates dropped : 19


In [40]:
# Save the processed dataset
df.to_csv('../data/processed/processed_reviews.csv', index=False)

In [41]:
#save the real data 
df_finetune.to_csv('../data/processed/processed_real_reviews.csv', index=False)

SPLITING THE DATASET FOR TRAIN, VAL AND TEST DATA


In [42]:

# ── Step 1: Create stratify key ───────────────────────────────
# FOCUSING ON LABEL AND LANGUAGE STYLE FOR STRATIFICATION
df["stratify_key"] = (
    df["label"].astype(str) + "_" + 
    df["language_style"]
)
df.stratify_key.value_counts()

stratify_key
1_code_mixed          966
0_code_mixed          953
1_english             737
0_english             641
0_romanized_nepali    590
1_romanized_nepali    587
Name: count, dtype: int64

In [43]:
# ── Create stratify key ───────────────────────────────────────
df_finetune["stratify_key"] = (
    df_finetune["label"].astype(str) + "_" +
    df_finetune["language_style"]
)
print("Stratify key counts:")
print(df_finetune["stratify_key"].value_counts())

Stratify key counts:
stratify_key
1_code_mixed          444
0_code_mixed          359
0_english             339
1_english             262
0_romanized_nepali    257
1_romanized_nepali    227
Name: count, dtype: int64


In [44]:
# ── Step 2: First split — train vs temp (val+test) ────────────
syn_train, syn_temp = train_test_split(
    df,
    test_size    = 0.2,          # 80% train, 20% temp
    stratify     = df["stratify_key"],
    random_state = 42
)

# ── Step 3: Second split — val vs test ────────────────────────
syn_val, syn_test = train_test_split(
    syn_temp,
    test_size    = 0.5,          # 50% val, 50% test
    stratify     = syn_temp["stratify_key"],
    random_state = 42
)

In [45]:
# ── Split 60/20/20 ────────────────────────────────────────────
# Smaller dataset → give more to val/test
real_train, real_temp = train_test_split(
    df_finetune,
    test_size    = 0.4,
    stratify     = df_finetune["stratify_key"],
    random_state = 42
)
real_val, real_test = train_test_split(
    real_temp,
    test_size    = 0.5,
    stratify     = real_temp["stratify_key"],
    random_state = 42
)


# ── Verify ────────────────────────────────────────────────────
print(f"\n{'='*45}")
print(f"  FINETUNE SPLIT SUMMARY")
print(f"{'='*45}")
print(f"  Total : {len(df_finetune)}")
print(f"  Train : {len(real_train)}  ({len(real_train)/len(df_finetune)*100:.1f}%)")
print(f"  Val   : {len(real_val)}    ({len(real_val)/len(df_finetune)*100:.1f}%)")
print(f"  Test  : {len(real_test)}   ({len(real_test)/len(df_finetune)*100:.1f}%)")

print(f"\n  Label dist — Train:")
print(f"  {real_train['label'].value_counts().to_dict()}")
print(f"\n  Label dist — Val:")
print(f"  {real_val['label'].value_counts().to_dict()}")
print(f"\n  Label dist — Test:")
print(f"  {real_test['label'].value_counts().to_dict()}")


  FINETUNE SPLIT SUMMARY
  Total : 1888
  Train : 1132  (60.0%)
  Val   : 378    (20.0%)
  Test  : 378   (20.0%)

  Label dist — Train:
  {0: 573, 1: 559}

  Label dist — Val:
  {0: 191, 1: 187}

  Label dist — Test:
  {0: 191, 1: 187}


In [46]:
syn_train.language_style

3782             english
1493          code_mixed
3642          code_mixed
4174    romanized_nepali
2424          code_mixed
              ...       
988     romanized_nepali
480           code_mixed
1477             english
3687          code_mixed
2566             english
Name: language_style, Length: 3579, dtype: str

In [47]:
# ── Step 4: Verify proportions ────────────────────────────────
print(f"\nTotal  : {len(df)}")
print(f"Train  : {len(syn_train)}  ({len(syn_train)/len(df)*100:.1f}%)")
print(f"Val    : {len(syn_val)}    ({len(syn_val)/len(df)*100:.1f}%)")
print(f"Test   : {len(syn_test)}   ({len(syn_test)/len(df)*100:.1f}%)")

print("\n--- Label distribution check ---")
for split_name, split_df in [("Train", syn_train), ("Val", syn_val), ("Test", syn_test)]:
    dist = split_df["label"].value_counts(normalize=True).mul(100).round(1)
    print(f"{split_name}: {dist.to_dict()}")

print("\n--- Language style check ---")
for split_name, split_df in [("Train", syn_train), ("Val", syn_val), ("Test", syn_test)]:
    dist = split_df["language_style"].value_counts(normalize=True).mul(100).round(1)
    print(f"{split_name}: {dist.to_dict()}")


Total  : 4474
Train  : 3579  (80.0%)
Val    : 447    (10.0%)
Test   : 448   (10.0%)

--- Label distribution check ---
Train: {1: 51.2, 0: 48.8}
Val: {1: 51.2, 0: 48.8}
Test: {1: 51.1, 0: 48.9}

--- Language style check ---
Train: {'code_mixed': 42.9, 'english': 30.8, 'romanized_nepali': 26.3}
Val: {'code_mixed': 42.7, 'english': 30.9, 'romanized_nepali': 26.4}
Test: {'code_mixed': 43.1, 'english': 30.8, 'romanized_nepali': 26.1}


In [48]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Separate encoders for each feature ───────────────────────
le_lang   = LabelEncoder()
le_domain = LabelEncoder()
le_len    = LabelEncoder()

# ════════════════════════════════════════
# SYNTHETIC DATA
# ════════════════════════════════════════

# language_style
le_lang.fit(syn_train["language_style"])
syn_train["language_enc"] = le_lang.transform(syn_train["language_style"])
syn_val["language_enc"]   = le_lang.transform(syn_val["language_style"])
syn_test["language_enc"]  = le_lang.transform(syn_test["language_style"])

# length_type
le_len.fit(syn_train["length_type"])
syn_train["length_enc"] = le_len.transform(syn_train["length_type"])
syn_val["length_enc"]   = le_len.transform(syn_val["length_type"])
syn_test["length_enc"]  = le_len.transform(syn_test["length_type"])

# domain ← use le_domain, save to domain_enc
le_domain.fit(syn_train["domain"])
syn_train["domain_enc"] = le_domain.transform(syn_train["domain"])
syn_val["domain_enc"]   = le_domain.transform(syn_val["domain"])
syn_test["domain_enc"]  = le_domain.transform(syn_test["domain"])

# StandardScaler
scale_cols = [
    "word_count", "char_count", "avg_word_len",
    "slang_count", "exclamation_count", "uppercase_ratio"
]
scaler_syn = StandardScaler()
scaler_syn.fit(syn_train[scale_cols])
syn_train[scale_cols] = scaler_syn.transform(syn_train[scale_cols])
syn_val[scale_cols]   = scaler_syn.transform(syn_val[scale_cols])
syn_test[scale_cols]  = scaler_syn.transform(syn_test[scale_cols])

# ════════════════════════════════════════
# REAL/FINETUNE DATA
# ════════════════════════════════════════
# Note: domain was dropped from real data
# so use separate encoders for real

le_lang_ft = LabelEncoder()
le_len_ft  = LabelEncoder()

# language_style
le_lang_ft.fit(real_train["language_style"])
real_train["language_enc"] = le_lang_ft.transform(real_train["language_style"])
real_val["language_enc"]   = le_lang_ft.transform(real_val["language_style"])
real_test["language_enc"]  = le_lang_ft.transform(real_test["language_style"])

# length_type
le_len_ft.fit(real_train["length_type"])
real_train["length_enc"] = le_len_ft.transform(real_train["length_type"])
real_val["length_enc"]   = le_len_ft.transform(real_val["length_type"])
real_test["length_enc"]  = le_len_ft.transform(real_test["length_type"])

# StandardScaler — separate for real data
scaler_real = StandardScaler()
scaler_real.fit(real_train[scale_cols])
real_train[scale_cols] = scaler_real.transform(real_train[scale_cols])
real_val[scale_cols]   = scaler_real.transform(real_val[scale_cols])
real_test[scale_cols]  = scaler_real.transform(real_test[scale_cols])

# ════════════════════════════════════════
# META FEATURES
# ════════════════════════════════════════
# Separate lists since real data has no domain

syn_meta_features = [
    "word_count", "char_count", "avg_word_len",
    "slang_count", "exclamation_count", "question_count",
    "uppercase_ratio", "unique_word_ratio",
    "has_slang", "rating",
    "language_enc", "domain_enc", "length_enc"   # ← has domain
]

real_meta_features = [
    "word_count", "char_count", "avg_word_len",
    "slang_count", "exclamation_count", "question_count",
    "uppercase_ratio", "unique_word_ratio",
    "has_slang", "rating",
    "language_enc", "length_enc"                 # ← no domain
]



In [49]:
#Vocab (for deep learning models)
from collections import Counter
combined = pd.concat([
    syn_train["review_text"],
    real_train["review_text"]
])

def build_vocab(texts, max_vocab=10000):
    words  = [w for text in texts for w in text.lower().split()]
    counts = Counter(words)
    vocab  = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counts.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab  = build_vocab(syn_train["review_text"])
vocab_final = build_vocab(combined)


In [50]:
# ── Text features ─────────────────────────────────────────────
# 1. TF-IDF (for traditional ML)
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Pretrain TF-IDF (syn_train only) ─────────────────────────
tfidf_syn = TfidfVectorizer(
    max_features = 5000,
    ngram_range  = (1, 2)
)
tfidf_syn.fit(syn_train["review_text"])     # ← syn_train only

# Transform all synthetic splits
X_syn_train_tfidf = tfidf_syn.transform(syn_train["review_text"])
X_syn_val_tfidf   = tfidf_syn.transform(syn_val["review_text"])
X_syn_test_tfidf  = tfidf_syn.transform(syn_test["review_text"])

print(f"Syn TF-IDF shape : {X_syn_train_tfidf.shape}")

Syn TF-IDF shape : (3579, 5000)


In [51]:

# ── Finetune TF-IDF (real_train only) ────────────────────────
tfidf_real = TfidfVectorizer(
    max_features = 8000,     # higher — real vocab is larger
    ngram_range  = (1, 2)
)
tfidf_real.fit(real_train["review_text"])   # ← real_train only

# Transform all real splits
X_real_train_tfidf = tfidf_real.transform(real_train["review_text"])
X_real_val_tfidf   = tfidf_real.transform(real_val["review_text"])
X_real_test_tfidf  = tfidf_real.transform(real_test["review_text"])

print(f"Real TF-IDF shape : {X_real_train_tfidf.shape}")

Real TF-IDF shape : (1132, 8000)


In [52]:
cols_to_drop = [
    "stratify_key",   # ← leakage 🚨 must drop
    "domain",         # ← raw string, already encoded
    "language_style", # ← raw string, already encoded
    "length_type",    # ← raw string, already encoded
    "date",           # ← not used in model
]
# ── Drop helper column ────────────────────────────────────────
# ── Synthetic splits ──────────────────────────────────────────
for split in [syn_train, syn_val, syn_test]:
    drop = [c for c in cols_to_drop if c in split.columns]
    split.drop(columns=drop, inplace=True)

# ── Real splits ───────────────────────────────────────────────
for split in [real_train, real_val, real_test]:
    drop = [c for c in cols_to_drop if c in split.columns]
    split.drop(columns=drop, inplace=True)

SAVING THE PROCESS

In [53]:
import joblib
import pickle



# ── Save TF-IDF vectorizer ────────────────────────────────────
joblib.dump(tfidf_syn,  "../models/traditional/tfidf_syn.pkl")
joblib.dump(tfidf_real, "../models/traditional/tfidf_real.pkl")

# ── Save vocab (for deep NN) ──────────────────────────────────

with open("../models/deep_nn/vocab_syn.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open("../models/deep_nn/vocab_final.pkl", "wb") as f:
    pickle.dump(vocab_final, f)


# ── Save feature column list ──────────────────────────────────
with open("../models/meta_features.pkl", "wb") as f:
    pickle.dump(syn_meta_features, f)

print("✅ All preprocessing artifacts saved!")

✅ All preprocessing artifacts saved!


In [54]:
# Synthetic splits
syn_train.to_csv("../data/processed/syn_train.csv", index=False)
syn_val.to_csv("../data/processed/syn_val.csv",     index=False)
syn_test.to_csv("../data/processed/syn_test.csv",   index=False)

# Real/finetune splits
real_train.to_csv("../data/processed/real_train.csv", index=False)
real_val.to_csv("../data/processed/real_val.csv",     index=False)
real_test.to_csv("../data/processed/real_test.csv",   index=False)